# LeWM Push-T: Prescribed vs Free — PIXELS

Полный эксперимент: CNN encoder учит embedding из 96×96 пикселей (free)
vs prescribed axes (block_x, block_y, block_angle) из state.

**Runtime → Change runtime type → T4 GPU**, потом **Runtime → Run all**.

~1-2 часа.

In [ ]:
!pip install h5py huggingface_hub -q

## 1. Скачиваем данные LeWM Push-T (13 GB → ~4 GB распакованный)

In [ ]:
import os

H5_PATH = '/content/pusht_expert_train.h5'

if not os.path.exists(H5_PATH):
    print('Downloading dataset...')
    !huggingface-cli download quentinll/lewm-pusht pusht_expert_train.h5.zst --repo-type dataset --local-dir /content/
    print('Decompressing...')
    !apt-get install -y zstd -qq
    !zstd -d /content/pusht_expert_train.h5.zst -o {H5_PATH}
    !rm /content/pusht_expert_train.h5.zst
    print('Done!')
else:
    print(f'Dataset already exists: {H5_PATH}')

!ls -lh {H5_PATH}

In [ ]:
# Изучаем структуру HDF5
import h5py
import numpy as np

with h5py.File(H5_PATH, 'r') as f:
    print('Top-level keys:', list(f.keys()))
    # Обычно: episodes/episode_0/pixels, action, state, proprio
    # или flat: pixels, action, state
    def explore(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f'  {name}: shape={obj.shape}, dtype={obj.dtype}')
    f.visititems(explore)

## 2. Dataset

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
H = 3       # history size (как в LeWM)
FRAMESKIP = 5  # как в LeWM конфиге

class PushTPixelDataset(Dataset):
    """
    Загружает Push-T HDF5, нарезает на окна (H+2) кадров.
    Возвращает pixels (H+2, C, 96, 96), actions (H+1, 2), states (H+2, 5).
    
    Адаптируется к структуре HDF5 автоматически.
    """
    def __init__(self, h5_path, H=3, frameskip=5, max_episodes=None):
        self.H = H
        self.windows = []
        
        with h5py.File(h5_path, 'r') as f:
            keys = list(f.keys())
            
            # Detect structure
            if 'pixels' in keys:
                # Flat structure: pixels (N, T, H, W, C), etc.
                self._load_flat(f, H, frameskip, max_episodes)
            else:
                # Episode structure: episode_0/pixels, etc.
                ep_keys = sorted([k for k in keys if 'episode' in k or k.isdigit()],
                                key=lambda x: int(''.join(filter(str.isdigit, x)) or 0))
                if not ep_keys:
                    # Maybe nested differently
                    ep_keys = sorted(keys)
                self._load_episodes(f, ep_keys, H, frameskip, max_episodes)
        
        print(f'Dataset: {len(self.windows)} windows')
    
    def _load_flat(self, f, H, frameskip, max_episodes):
        """Flat structure: pixels shape (N_episodes, T_steps, H, W, C)"""
        pixels = f['pixels']
        actions = f['action']
        states = f['state'] if 'state' in f else None
        
        n_ep = pixels.shape[0]
        if max_episodes:
            n_ep = min(n_ep, max_episodes)
        
        for i in range(n_ep):
            pix = pixels[i]     # (T, H, W, C)
            act = actions[i]    # (T, action_dim)
            st = states[i] if states is not None else None  # (T, state_dim)
            
            # Apply frameskip
            T = pix.shape[0]
            idx = list(range(0, T, frameskip))
            pix = pix[idx]
            act = act[idx]
            if st is not None:
                st = st[idx]
            
            T2 = len(idx)
            seq_len = H + 2
            
            for t in range(T2 - seq_len):
                p = pix[t:t+seq_len]  # (H+2, h, w, c)
                a = act[t:t+seq_len-1]  # (H+1, action_dim)
                s = st[t:t+seq_len] if st is not None else np.zeros((seq_len, 5))
                self.windows.append((p, a, s))
            
            if (i+1) % 50 == 0:
                print(f'  Loaded {i+1}/{n_ep} episodes, {len(self.windows)} windows')
    
    def _load_episodes(self, f, ep_keys, H, frameskip, max_episodes):
        """Episode structure: episode_N/{pixels, action, state}"""
        n_ep = len(ep_keys)
        if max_episodes:
            n_ep = min(n_ep, max_episodes)
        
        for i in range(n_ep):
            ep = f[ep_keys[i]]
            
            # Find pixel key
            pix_key = next((k for k in ep.keys() if 'pixel' in k.lower()), None)
            act_key = next((k for k in ep.keys() if 'action' in k.lower()), None)
            st_key = next((k for k in ep.keys() if 'state' in k.lower()), None)
            
            if pix_key is None or act_key is None:
                continue
            
            pix = ep[pix_key][:]   # (T, H, W, C) or (T, C, H, W)
            act = ep[act_key][:]   # (T, action_dim)
            st = ep[st_key][:] if st_key else np.zeros((pix.shape[0], 5))
            
            # Apply frameskip
            T = pix.shape[0]
            idx = list(range(0, T, frameskip))
            pix = pix[idx]
            act = act[idx]
            st = st[idx]
            
            T2 = len(idx)
            seq_len = H + 2
            
            for t in range(T2 - seq_len):
                p = pix[t:t+seq_len]
                a = act[t:t+seq_len-1]
                s = st[t:t+seq_len]
                self.windows.append((p, a, s))
            
            if (i+1) % 50 == 0:
                print(f'  Loaded {i+1}/{n_ep} episodes, {len(self.windows)} windows')
    
    def __len__(self):
        return len(self.windows)
    
    def __getitem__(self, idx):
        pix, act, state = self.windows[idx]
        
        # pixels: (H+2, h, w, c) → (H+2, c, h, w), float, normalize
        pix = torch.from_numpy(pix).float()
        if pix.shape[-1] in [1, 3]:  # (T, H, W, C) → (T, C, H, W)
            pix = pix.permute(0, 3, 1, 2)
        pix = pix / 255.0
        
        act = torch.from_numpy(act).float()
        state = torch.from_numpy(state).float()
        
        return pix, act, state

In [ ]:
# Загружаем (может занять пару минут)
ds = PushTPixelDataset(H5_PATH, H=H, frameskip=FRAMESKIP)

# Проверяем
pix, act, state = ds[0]
print(f'Pixels: {pix.shape}, range [{pix.min():.2f}, {pix.max():.2f}]')
print(f'Action: {act.shape}')
print(f'State:  {state.shape}')
print(f'State[0]: {state[0].numpy()}')
print(f'  → block_x={state[0,2]:.1f}, block_y={state[0,3]:.1f}, angle={state[0,4]:.3f}')

In [ ]:
# Train/val split
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

n_train = int(len(ds) * 0.9)
n_val = len(ds) - n_train
train_ds, val_ds = random_split(ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(SEED))

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, 
                      drop_last=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=64, shuffle=False,
                    num_workers=2, pin_memory=True)

print(f'Train: {n_train}, Val: {n_val}')

## 3. Models

In [ ]:
# SIGReg из LeWM
class SIGReg(nn.Module):
    def __init__(self, knots=17, num_proj=512):
        super().__init__()
        self.num_proj = num_proj
        t = torch.linspace(0, 3, knots)
        dt = 3 / (knots - 1)
        w = torch.full((knots,), 2*dt); w[[0,-1]] = dt
        phi = torch.exp(-t.square() / 2.0)
        self.register_buffer('t', t)
        self.register_buffer('phi', phi)
        self.register_buffer('weights', w * phi)

    def forward(self, proj):
        A = torch.randn(proj.size(-1), self.num_proj, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        return ((err @ self.weights) * proj.size(-2)).mean()


class CNNEncoder(nn.Module):
    """CNN encoder: 96x96x3 → D-dim embedding (per frame)"""
    def __init__(self, D=3, in_channels=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 4, stride=2, padding=1),  # 48x48
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),           # 24x24
            nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),          # 12x12
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),         # 6x6
            nn.BatchNorm2d(256), nn.GELU(),
            nn.AdaptiveAvgPool2d(1),                             # 1x1
        )
        self.proj = nn.Sequential(
            nn.Linear(256, 128), nn.GELU(),
            nn.Linear(128, D)
        )
    
    def forward(self, pixels, states=None):
        """
        pixels: (B, T, C, H, W)
        Returns: (B, T, D)
        """
        B, T = pixels.shape[:2]
        x = pixels.reshape(B * T, *pixels.shape[2:])  # (B*T, C, H, W)
        x = self.conv(x).squeeze(-1).squeeze(-1)       # (B*T, 256)
        x = self.proj(x)                                # (B*T, D)
        return x.reshape(B, T, -1)


class PrescribedEncoder(nn.Module):
    """state[2:5] → normalized (block_x/512, block_y/512, angle/2π)"""
    def __init__(self):
        super().__init__()
        self.register_buffer('scale',
            torch.tensor([1.0/512, 1.0/512, 1.0/(2*3.14159265)]))
    
    def forward(self, pixels, states):
        """Ignores pixels, uses states."""
        return states[..., 2:5] * self.scale


class ActionEncoder(nn.Module):
    def __init__(self, d_in=2, d_out=3, h=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, h), nn.GELU(), nn.Linear(h, d_out))
    def forward(self, a):
        return self.net(a)


class Predictor(nn.Module):
    def __init__(self, D=3, H=3, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(H * 2 * D, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, D))
    def forward(self, emb, act_emb):
        x = torch.cat([emb, act_emb], dim=-1).reshape(emb.size(0), -1)
        return self.net(x)


print('Models defined.')

## 4. Training

In [ ]:
def run_pixel_experiment(mode, train_dl, val_dl, D=3, H=3, epochs=50,
                         lr=3e-4, lam=0.09, use_sigreg=True, seed=42):
    """
    mode: 'prescribed' or 'free_cnn'
    """
    torch.manual_seed(seed)
    
    if mode == 'prescribed':
        encoder = PrescribedEncoder().to(device)
    else:
        # Detect channels from data
        sample_pix = next(iter(train_dl))[0]
        in_ch = sample_pix.shape[2]  # (B, T, C, H, W)
        encoder = CNNEncoder(D=D, in_channels=in_ch).to(device)
    
    act_enc = ActionEncoder(d_in=2, d_out=D, h=32).to(device)
    predictor = Predictor(D=D, H=H, hidden=128).to(device)
    sigreg = SIGReg(17, 512).to(device) if use_sigreg else None
    
    params = list(encoder.parameters()) + list(act_enc.parameters()) + list(predictor.parameters())
    n_params = sum(p.numel() for p in params if p.requires_grad)
    opt = torch.optim.AdamW([p for p in params if p.requires_grad],
                            lr=lr, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    
    print(f'\n--- {mode} (params={n_params:,}, D={D}, sigreg={use_sigreg}) ---')
    
    history = []
    best_vp, best_ep = float('inf'), 0
    
    for ep in range(1, epochs + 1):
        t0 = time.time()
        
        # --- Train ---
        encoder.train(); act_enc.train(); predictor.train()
        train_pl, train_sl, n = 0, 0, 0
        
        for pix, act, state in train_dl:
            pix = pix.to(device)
            act = act.to(device)
            state = state.to(device)
            
            emb = encoder(pix, state)       # (B, H+2, D)
            ctx = emb[:, :H]                 # (B, H, D)
            tgt = emb[:, H]                  # (B, D)
            ae = act_enc(act[:, :H])          # (B, H, D)
            pred = predictor(ctx, ae)         # (B, D)
            
            pl = F.mse_loss(pred, tgt.detach())
            sl = sigreg(emb.transpose(0, 1)) if sigreg else torch.tensor(0.0)
            loss = pl + lam * sl
            
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            
            b = pix.size(0)
            train_pl += pl.item() * b
            train_sl += sl.item() * b
            n += b
        
        train_pl /= n
        train_sl /= n
        
        # --- Val ---
        encoder.eval(); act_enc.eval(); predictor.eval()
        val_pl, val_n = 0, 0
        
        with torch.no_grad():
            for pix, act, state in val_dl:
                pix = pix.to(device)
                act = act.to(device)
                state = state.to(device)
                
                emb = encoder(pix, state)
                ctx = emb[:, :H]
                tgt = emb[:, H]
                ae = act_enc(act[:, :H])
                pred = predictor(ctx, ae)
                
                val_pl += F.mse_loss(pred, tgt).item() * pix.size(0)
                val_n += pix.size(0)
        
        vp = val_pl / val_n
        if vp < best_vp:
            best_vp, best_ep = vp, ep
        
        sch.step()
        elapsed = time.time() - t0
        history.append({'ep': ep, 'tp': train_pl, 'vp': vp, 'ts': train_sl})
        
        if ep % 5 == 0 or ep == 1:
            print(f'  ep {ep:3d} | train {train_pl:.6f} | '
                  f'val {vp:.6f} | sig {train_sl:.4f} | {elapsed:.1f}s')
    
    print(f'  BEST: {best_vp:.6f} (ep {best_ep})')
    return {
        'mode': mode, 'params': n_params, 'best_vp': best_vp,
        'best_ep': best_ep, 'history': history
    }

In [ ]:
# === MAIN RUN ===
results = {}

for mode in ['prescribed', 'free_cnn']:
    r = run_pixel_experiment(mode, train_dl, val_dl, D=3, H=H,
                             epochs=50, use_sigreg=True, seed=SEED)
    results[mode] = r

## 5. Результаты

In [ ]:
print('\n' + '='*60)
print('RESULTS — PIXELS'.center(60))
print('='*60)

for mode in ['prescribed', 'free_cnn']:
    r = results[mode]
    print(f"  {mode:15s}: best_vp={r['best_vp']:.6f}  "
          f"ep={r['best_ep']}  params={r['params']:,}")

p = results['prescribed']['best_vp']
f = results['free_cnn']['best_vp']
ratio = f / p if p > 0 else 0
print(f'\n  Ratio free/prescribed = {ratio:.1f}x')
print(f'  {"Prescribed лучше" if p < f else "Free лучше"}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
colors = {'prescribed': '#2ecc71', 'free_cnn': '#e74c3c'}

for mode in ['prescribed', 'free_cnn']:
    h = results[mode]['history']
    ax.plot([x['ep'] for x in h], [x['vp'] for x in h],
            label=f"{mode} ({results[mode]['params']:,} params)",
            color=colors[mode], linewidth=2)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Val pred_loss', fontsize=12)
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_title('LeWM Push-T: Prescribed vs Free CNN (pixels)', fontsize=14)

plt.tight_layout()
plt.savefig('lewm_pixels_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Ablation: без SIGReg

In [ ]:
print('Ablation: without SIGReg')
ablation = {}
for mode in ['prescribed', 'free_cnn']:
    r = run_pixel_experiment(mode, train_dl, val_dl, D=3, H=H,
                             epochs=50, use_sigreg=False, seed=SEED)
    ablation[mode] = r
    print(f'  {mode:15s} no_sigreg: {r["best_vp"]:.6f}')
    print(f'  {mode:15s} w/ sigreg: {results[mode]["best_vp"]:.6f}')

## 7. Скачать результаты

In [ ]:
import json, zipfile
from google.colab import files

# Собираем всё в JSON
output = {
    'main': {k: {kk: vv for kk, vv in v.items() if kk != 'history'}
             for k, v in results.items()},
    'ablation': {k: {kk: vv for kk, vv in v.items() if kk != 'history'}
                 for k, v in ablation.items()},
    'history': {k: v['history'] for k, v in results.items()},
}

with open('lewm_pixels_results.json', 'w') as f:
    json.dump(output, f, indent=2)

# ZIP
with zipfile.ZipFile('lewm_pixels_all.zip', 'w') as z:
    z.write('lewm_pixels_results.json')
    z.write('lewm_pixels_results.png')

files.download('lewm_pixels_all.zip')
print('Done!')